# Preprocess

## 1. Fetch images from Surfdrive

In [ ]:
import os
import regex
import requests
from dotenv import load_dotenv
from pathlib import Path
from requests.auth import HTTPBasicAuth
from IPython.display import clear_output

load_dotenv()

In [ ]:
def squeal(text=None):
    clear_output(wait=True)
    if not text is None: print(text)

In [ ]:
USERNAME = os.getenv("SURFDRIVE_USERNAME") # SurfDrive login email address, stored in file .env
APP_PASSWD = os.getenv("SURFDRIVE_APP_PASSWD") # create on SurfDrive at User / Settings / Security
BASE_URL = f"https://surfdrive.surf.nl/remote.php/dav/files/{USERNAME}/"

In [ ]:
OFFICE = "Werkendam"
START_YEAR = "1924"
END_YEAR = "1927"
SUFFIX = "03"
DIRECTORY = f"rags2riches/bhic/{OFFICE}/deel_{START_YEAR}-{END_YEAR}/"

for file_nbr in range(500, 1248):
    filename = f"MFF-{OFFICE}-{START_YEAR}-{END_YEAR}-{SUFFIX}-{str(file_nbr).zfill(5)}.jpg"
    response = requests.get(BASE_URL + DIRECTORY + filename, auth=HTTPBasicAuth(USERNAME, APP_PASSWD))
    response.raise_for_status()
    with open(filename, "wb") as f:
        f.write(response.content)
    assert Path(filename).exists()
    squeal(file_nbr)

## 2. Split images and deskew

In [ ]:
from split_scans_v8 import process_dir

In [ ]:
DIR = "/home/erikt/projects/rags/memories_crawl/scans/bhic/Werkendam/deel_1924-1927"

process_dir(DIR)

## 3. Select images

In [ ]:
from image_selector_simple import image_selector_simple

In [ ]:
DIR = "/home/erikt/projects/rags/memories_crawl/scans/bhic/Werkendam/deel_1924-1927"
OUT_DIR = f"{DIR}/out_L"

image_selector_simple(OUT_DIR, start_index=1, mode="selected")

In [ ]:
import json
import shutil
import os
from pathlib import Path


def copy_selected_files_to_dir(source_dir=".", selected_json_path="selected.json", dest_dirname="selected"):
    """
    Copy every .jpg file listed in selected_json_path from source_dir into
    source_dir/dest_dirname. Missing files are reported, not raised, so
    one bad entry doesn't stop the rest of the batch.

    Returns (copied, missing): lists of filenames that were copied /
    could not be found in source_dir.
    """
    source_dir = Path(source_dir)
    dest_dir = source_dir / dest_dirname
    dest_dir.mkdir(exist_ok=True)

    with open(os.path.join(source_dir, selected_json_path), encoding="utf-8") as f:
        filenames = json.load(f)

    copied = []
    missing = []
    for name in filenames:
        src = source_dir / name
        if src.exists():
            shutil.copy2(src, dest_dir / name)
            copied.append(name)
        else:
            missing.append(name)

    print(f"Copied {len(copied)} file(s) to {dest_dir}")
    if missing:
        print(f"Warning: {len(missing)} file(s) not found: {missing}")

    return copied, missing

In [ ]:
copy_selected_files_to_dir(source_dir="../memories_crawl/scans/bhic/Werkendam/deel_1924-1927/out_L")